# InfiniteTalk auf 16 GB testen (Colab / Kaggle)

Dieses Notebook startet die **Gradio-App** (`app.py`) mit **fp8-Quantisierung**, damit
InfiniteTalk-14B auch auf GPUs mit wenig VRAM laufen kann.

**Was das Notebook macht**

1. Prueft, ob dein Runtime gross genug ist (VRAM / RAM / Platte)
2. Clont das Repo und installiert die Abhaengigkeiten
3. Laedt nur die Gewichte, die das gewaehlte Profil wirklich braucht
4. Startet die App und gibt dir eine oeffentliche `*.gradio.live` URL

**Speicherbedarf des 14B-Modells**

| Profil | Wo liegen die Gewichte | Bedarf |
|---|---|---|
| `bf16` | komplett auf der GPU | **>= 30 GiB VRAM** (z. B. A100 40GB) |
| `fp8-gpu` | fp8-DiT (14 GiB) bleibt auf der GPU | **>= 17 GiB VRAM**, RAM >= 8 GiB |
| `fp8-offload` | fp8-DiT + T5 werden auf die CPU ausgelagert | **RAM >= 21 GiB**, VRAM >= 4 GiB |

> **Wichtig:** Die kostenlosen T4 (Colab) und P100 (Kaggle) haben 16 GiB VRAM *und*
> nur 12-13 GiB RAM - das reicht fuer keins der drei Profile. Auf solch einem Runtime
> sagt dir die Preflight-Zelle das direkt. Nutze dann Colab Pro (A100), Modal oder
> eine eigene GPU mit 24 GB.
>
> Hinweis: `int8` ist **keine** Alternative - im InfiniteTalk-Repo existiert nur ein
> quantiertes T5 fuer `fp8`, `--quant int8` bricht deshalb beim Laden ab.

In [ ]:
import os, re, shutil, subprocess, sys, time

# ---------------------------------------------------------------------------
# Konfiguration - hier bei Bedarf anpassen
# ---------------------------------------------------------------------------
REPO_URL         = "https://github.com/Woolvayne/InfiniteTalk.git"
REPO_BRANCH      = "arena/01a0d417-infinitetalk"  # Branch mit dem Vercel/ASGI-Fix
MODE             = "single"    # 'single' = 1 Person, 'multi' = 2 Personen
DOWNLOAD_KOKORO  = True        # TTS-Stimmen (~350 MB), nur fuer den TTS-Modus

# Arbeitsverzeichnis: Colab -> /content, Kaggle -> /kaggle/working
BASE    = "/content" if os.path.isdir("/content") else "/kaggle/working"
if not os.path.isdir(BASE):          # lokaler Test / anderes Environment
    BASE = os.getcwd()
WORKDIR = os.path.join(BASE, "InfiniteTalk")
WEIGHTS = os.path.join(WORKDIR, "weights")
LOG     = os.path.join(BASE, "infinitetalk.log")

print(f"Platform : {BASE}")
print(f"Repo     : {WORKDIR}")
print(f"Gewichte : {WEIGHTS}")

In [ ]:
def gpu_vram_gib():
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.total",
             "--format=csv,noheader,nounits"], text=True)
        return int(out.strip().splitlines()[0]) / 1024
    except Exception:
        return 0.0

def system_ram_gib():
    try:
        with open("/proc/meminfo") as f:
            kb = int(next(l for l in f if l.startswith("MemTotal")).split()[1])
        return kb / 1024 ** 2
    except Exception:
        return 0.0

vram, ram = gpu_vram_gib(), system_ram_gib()
disk = shutil.disk_usage(BASE).free / 1024 ** 3
print(f"GPU-VRAM  : {vram:6.1f} GiB")
print(f"System-RAM: {ram:6.1f} GiB")
print(f"Frei      : {disk:6.1f} GiB")

# Profil automatisch waehlen: VRAM zuerst (schnellste Option gewinnt),
# CPU-Offload nur als letzte Moeglichkeit fuer wenig VRAM / viel RAM.
if vram >= 30:
    PROFILE, QUANT, EXTRA = "bf16", None, []
elif vram >= 17:
    PROFILE, QUANT, EXTRA = "fp8-gpu", "fp8", ["--offload_model", "False"]
elif ram >= 21:
    PROFILE, QUANT, EXTRA = "fp8-offload", "fp8", ["--num_persistent_param_in_dit", "0"]
else:
    PROFILE, QUANT, EXTRA = "too-small", None, []

NEEDED_DISK = {"bf16": 45, "fp8-offload": 22, "fp8-gpu": 22}.get(PROFILE, 0)
print(f"\n--> Profil: {PROFILE}")

if PROFILE == "too-small":
    print("""
ACHTUNG - dieser Runtime ist zu klein fuer InfiniteTalk-14B.

Du brauchst ENTWEDER
  * mindestens 17 GiB VRAM  (fp8-Gewichte bleiben auf der GPU), ODER
  * mindestens 21 GiB RAM   (fp8-Gewichte + T5 werden auf die CPU ausgelagert).

Kostenlose Runtimes (Colab T4 / Kaggle P100: 16 GiB VRAM, 12-13 GiB RAM)
schaffen beides nicht. Moeglichkeiten:
  * Colab Pro      -> A100 40GB  (~10 EUR/Monat)
  * Modal          -> $30 Gratis-Credits/Monat, A100 40GB
  * eigene GPU     -> ab 24 GB (RTX 3090/4090) mit fp8
""")
elif NEEDED_DISK > disk:
    print(f"ACHTUNG: {NEEDED_DISK} GiB Platz noetig, aber nur {disk:.0f} GiB frei.")

In [ ]:
# ---- Repo klonen (ueberspringen, wenn es schon existiert) -----------------
if os.path.isdir(os.path.join(WORKDIR, "app.py")):
    print("Repo existiert bereits - Clone uebersprungen")
else:
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, "--depth", "1",
                    REPO_URL, WORKDIR], check=True)

# ---- Abhaengigkeiten -------------------------------------------------------
# decord wird nur von der VACE-Pipeline gebraucht (nicht von InfiniteTalk) und
# baut auf neuen Python-Versionen oft nicht durch -> separat und toleriert.
lines = [l.strip() for l in open(os.path.join(WORKDIR, "requirements.txt"))
         if l.strip() and not l.startswith("#")]
core = [l for l in lines if not l.lower().startswith("decord")]
open(os.path.join(WORKDIR, "requirements.core.txt"), "w").write("\n".join(core) + "\n")

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                os.path.join(WORKDIR, "requirements.core.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "huggingface_hub"], check=True)
print("Basis-Abhaengigkeiten installiert")

try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "decord"], check=True)
    print("decord installiert")
except Exception:
    print("Hinweis: decord konnte nicht gebaut werden. Wird nur fuer die "
          "VACE-Pipeline gebraucht - InfiniteTalk laeuft trotzdem.")

In [ ]:
from huggingface_hub import snapshot_download, hf_hub_download

def dl(repo, local, allow=None):
    print(f"-> {repo}")
    snapshot_download(repo, local_dir=local, allow_patterns=allow)

# Audio-Encoder wird immer gebraucht
dl("TencentGameMate/chinese-wav2vec2-base",
   f"{WEIGHTS}/chinese-wav2vec2-base")
hf_hub_download("TencentGameMate/chinese-wav2vec2-base", "model.safetensors",
                revision="refs/pr/1",
                local_dir=f"{WEIGHTS}/chinese-wav2vec2-base")

if QUANT:
    # fp8: vom Base-Modell nur die kleinen Dateien - der 14B-DiT kommt
    # quantisiert aus dem InfiniteTalk-Repo (14 GiB statt 28 GiB).
    dl("Wan-AI/Wan2.1-I2V-14B-480P", f"{WEIGHTS}/Wan2.1-I2V-14B-480P",
       allow=["config.json", "Wan2.1_VAE.pth",
              "models_clip_open-clip-xlm-roberta-large-vit-huge-14.pth",
              "google/umt5-xxl/*", "xlm-roberta-large/*"])
    dl("MeiGen-AI/InfiniteTalk", f"{WEIGHTS}/InfiniteTalk",
       allow=["quant_models/*", f"{MODE}/infinitetalk.safetensors"])
else:
    # bf16: komplettes Base-Modell (~40 GiB)
    dl("Wan-AI/Wan2.1-I2V-14B-480P", f"{WEIGHTS}/Wan2.1-I2V-14B-480P")
    dl("MeiGen-AI/InfiniteTalk", f"{WEIGHTS}/InfiniteTalk")

if DOWNLOAD_KOKORO:
    dl("hexgrad/Kokoro-82M", f"{WEIGHTS}/Kokoro-82M")

print("\nDownload fertig.")

In [ ]:
# ---- Kontrolle, ob alles da ist -------------------------------------------
need = [f"{WEIGHTS}/chinese-wav2vec2-base",
        f"{WEIGHTS}/Wan2.1-I2V-14B-480P/config.json",
        f"{WEIGHTS}/Wan2.1-I2V-14B-480P/Wan2.1_VAE.pth",
        f"{WEIGHTS}/InfiniteTalk/{MODE}/infinitetalk.safetensors"]
if QUANT:
    need += [f"{WEIGHTS}/InfiniteTalk/quant_models/infinitetalk_{MODE}_{QUANT}.safetensors",
             f"{WEIGHTS}/InfiniteTalk/quant_models/infinitetalk_{MODE}_{QUANT}.json",
             f"{WEIGHTS}/InfiniteTalk/quant_models/t5_{QUANT}.safetensors",
             f"{WEIGHTS}/InfiniteTalk/quant_models/t5_map_{QUANT}.json"]

missing = [p for p in need if not os.path.exists(p)]
print("Fehlende Dateien:", missing or "keine - alle Gewichte vorhanden")

In [ ]:
# ---- App starten ----------------------------------------------------------
cmd = [sys.executable, "app.py",
       "--ckpt_dir", f"{WEIGHTS}/Wan2.1-I2V-14B-480P",
       "--wav2vec_dir", f"{WEIGHTS}/chinese-wav2vec2-base",
       "--infinitetalk_dir", f"{WEIGHTS}/InfiniteTalk/{MODE}/infinitetalk.safetensors",
       "--motion_frame", "9"]
if QUANT:
    cmd += ["--quant", QUANT,
            "--quant_dir",
            f"{WEIGHTS}/InfiniteTalk/quant_models/infinitetalk_{MODE}_{QUANT}.safetensors"]
cmd += EXTRA

env = os.environ.copy()
env["GRADIO_SHARE"] = "1"   # oeffentliche *.gradio.live URL
env["GRADIO_DEBUG"] = "0"   # kein Reload-Watcher im Notebook

log = open(LOG, "w")
proc = subprocess.Popen(cmd, cwd=WORKDIR, env=env,
                        stdout=log, stderr=subprocess.STDOUT)
print(f"Server gestartet (PID {proc.pid}), Log: {LOG}")
print("Das Laden der Gewichte dauert einige Minuten ...")

url = None
for _ in range(240):                      # bis zu 8 Minuten warten
    time.sleep(2)
    if proc.poll() is not None:
        break
    try:
        m = re.search(r"https://[a-z0-9-]+\.gradio\.live", open(LOG).read())
    except Exception:
        continue
    if m:
        url = m.group(0)
        break

print()
if url:
    print("Oeffentliche URL (gueltig fuer 72 Stunden):")
    print("   ", url)
else:
    print("Keine Share-URL gefunden - letzte Log-Zeilen:")
    print("".join(open(LOG).readlines()[-40:]))

## Troubleshooting

**Server beenden**
```python
proc.terminate()
```

**Kein Share-Link?** Dann lief der Start schief. Log ansehen:
```python
print("".join(open(LOG).readlines()[-60:]))
```

**`CUDA out of memory`**
* Profil `fp8-gpu` (Gewichte auf der GPU): `--frame_num` reduzieren, z. B. `--frame_num 41`
  statt 81 - das ist der wirksamste Hebel, weil die Aktivirungen schrumpfen.
* Profil `fp8-offload`: braucht ~21 GiB RAM. Wenn der Runtime weniger hat, kein Profil
  mit Offload benutzen.
* `--sample_steps` sind bereits auf 8 voreingestellt - hoehere Werte brauchen Zeit,
  nicht mehr VRAM.

**`FileNotFoundError` beim Laden von T5/DiT**
Dann fehlt eine quantisierte Datei. Pruefe, dass `quant_models/` komplett ist
(vorherige Zelle zeigt fehlende Dateien an). `--quant int8` funktioniert nicht -
es gibt nur `t5_fp8.safetensors`.

**App ist langsam**
Auf einer T4/P100 laeuft fp8 ohne die schnellen Marlin-Kernel, also deutlich
langsamer als auf einer Ada/Hopper-GPU. Eine A100 40GB ist hier der sweet spot.

**Naechster Schritt: dauerhaft hosten**
Wenn der Test klappt: dasselbe Setup mit `modal deploy` (A100 40GB, $30 Gratis-Credits
pro Monat) laufen lassen - die App exportiert dafuer bereits ein ASGI-`app`-Objekt.